In [2]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers , models
from tensorflow.keras.utils import plot_model
import matplotlib.pyplot as plt
import glob
from PIL import Image
import ipywidgets as widgets
from IPython.display import display
import matplotlib.image as mpimg
from sklearn.utils import shuffle
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint


In [3]:
filepath = 'dataset/'
filelist = sorted(glob.glob(filepath+'*jpg'))
print(len(filelist))
print(filelist[0])

1100
dataset\cat.0.jpg


In [4]:
def Normalize(img):
    if np.max(img) != np.min(img):
        return (img - np.min(img))/(np.max(img) - np.min(img))
    else:
        return img

In [5]:
cat_img = []
dog_img = []
for i in range(len(filelist)):
    filename = filelist[i].split('/')[-1]
    if "cat" in filename:
        img = Image.open(filelist[i]).convert('L').resize((128,128))
        img = np.array(img,dtype = np.uint8)
        img = Normalize(img)
        img = np.expand_dims(img,axis =2)
        cat_img.append(img)
    elif "dog" in filename:
        img = Image.open(filelist[i]).convert('L').resize((128,128))
        img = np.array(img,dtype = np.uint8)
        img = Normalize(img)
        img = np.expand_dims(img,axis =2)
        dog_img.append(img)
    else:
        print('unsopported type image', filelist[i])
cat_img = np.array(cat_img)
dog_img = np.array(dog_img)
print(np.shape(cat_img))
print(np.shape(dog_img))

(550, 128, 128, 1)
(550, 128, 128, 1)


In [6]:
def show_img(img_index):
    plt.imshow(cat_img[img_index,:,:,0], cmap='gray')
    plt.axis('off') 
    plt.show()

slider = widgets.IntSlider(value=0, min=0, max=len(cat_img)-1, step=1)
widgets.interactive(show_img, img_index=slider)

interactive(children=(IntSlider(value=0, description='img_index', max=549), Output()), _dom_classes=('widget-i…

In [7]:
def show_img(img_index):
    plt.imshow(dog_img[img_index,:,:,0],cmap = 'gray')
    plt.axis('off')
    plt.show()

slider = widgets.IntSlider(value=0,min=0,max=len(dog_img),step =1)
widgets.interactive(show_img,img_index = slider)
            

interactive(children=(IntSlider(value=0, description='img_index', max=550), Output()), _dom_classes=('widget-i…

In [8]:
dog_train = dog_img[0:350]
dog_val = dog_img[350:450]
dog_test = dog_img[450:]

dog_train_label = np.zeros(len(dog_train), dtype = np.uint8)
dog_val_label = np.zeros(len(dog_val),dtype = np.uint8)
dog_test_label = np.zeros(len(dog_test), dtype = np.uint8)


cat_train = cat_img[:350]
cat_val = cat_img[350:450]
cat_test = cat_img[450:]

cat_train_label = np.ones(len(cat_train),dtype = np.uint8)
cat_val_label = np.ones(len(cat_val), dtype = np.uint8)
cat_test_label = np.ones(len(cat_test), dtype = np.uint8)

train_image = np.concatenate((dog_train,cat_train), axis = 0)
val_image = np.concatenate((dog_val,cat_val),axis = 0)
test_image = np.concatenate((dog_test,cat_test),axis = 0)

train_label = np.concatenate((dog_train_label,cat_train_label),axis =0)
val_label = np.concatenate((dog_val_label,cat_val_label), axis = 0)
test_label = np.concatenate((dog_test_label,cat_test_label),axis=0)

print("Shape of Train image:", np.shape(train_image), np.shape(train_label))
print("Shape of Val image:", np.shape(val_image), np.shape(val_label))
print("Shape of Test image:", np.shape(test_image),np.shape(test_label))


Shape of Train image: (700, 128, 128, 1) (700,)
Shape of Val image: (200, 128, 128, 1) (200,)
Shape of Test image: (200, 128, 128, 1) (200,)


In [9]:
def show_img(img_index):
    plt.imshow(train_image[img_index,:,:,0],cmap='gray')
    plt.axis("off")
    plt.show()

slider = widgets.IntSlider(value = 0, min =0, max=len(train_image)-1,step =1) 
widgets.interactive(show_img,img_index = slider)

interactive(children=(IntSlider(value=0, description='img_index', max=699), Output()), _dom_classes=('widget-i…

In [10]:
zipped = list(zip(train_image,train_label))
shuffled= shuffle(zipped,random_state=42)
unzipped = list(zip(*shuffled))
train_img = np.array(unzipped[0])
train_label = np.array(unzipped[1])
print("Shuffled train size and label:", np.shape(train_img),np.shape(train_label))

Shuffled train size and label: (700, 128, 128, 1) (700,)


In [11]:
def show_img(img_index):
    plt.imshow(train_img[img_index,:,:,0],cmap = 'gray')
    plt.axis('off')
    plt.show()
slider = widgets.IntSlider(value = 0, min = 0, max = len(train_img)-1, step = 1) 
widgets.interactive(show_img, img_index = slider)

interactive(children=(IntSlider(value=0, description='img_index', max=699), Output()), _dom_classes=('widget-i…

In [12]:
# Create our Second CNN model
inputs = layers.Input(shape=(128, 128, 1), name ='Input_layer')

# First Convolutional Block
x = layers.Conv2D(filters = 10, kernel_size= (3, 3), strides=(1,1),
                  padding='same', activation='relu', name ='Block1_CNN')(inputs)
x = layers.MaxPooling2D(pool_size=(2, 2), strides=None,
                        padding="valid" , name ='Block1_MaxPool')(x)

# Second Convolutional Block
x = layers.Conv2D(filters = 20, kernel_size= (3, 3), strides=(1,1),
                  padding='same', activation='relu', name ='Block2_CNN')(x)
x = layers.MaxPooling2D(pool_size=(2, 2), strides=None, padding="valid" )(x)


# Third Convolutional Block
x = layers.Conv2D(filters = 40, kernel_size= (3, 3), strides=(1,1),
                  padding='same', activation='relu', name ='Block3_CNN')(x)
x = layers.MaxPooling2D(pool_size=(2, 2), strides=None, padding="valid" )(x)


# Fourth Convolutional Block
x = layers.Conv2D(filters = 80, kernel_size= (3, 3), strides=(1,1),
                  padding='same', activation='relu', name ='Block4_CNN')(x)
x = layers.MaxPooling2D(pool_size=(2, 2), strides=None, padding="valid" )(x)


# Fifth Convolutional Block
x = layers.Conv2D(filters = 160, kernel_size= (3, 3), strides=(1,1),
                  padding='same', activation='relu', name ='Block5_CNN')(x)
x = layers.MaxPooling2D(pool_size=(2, 2), strides=None, padding="valid" )(x)


# Sixth Convolutional Block
x = layers.Conv2D(filters = 160, kernel_size= (3, 3), strides=(1,1),
                  padding='same', activation='relu', name ='Block6_CNN')(x)
x = layers.MaxPooling2D(pool_size=(2, 2), strides=None, padding="valid" )(x)

# Seventh Convolutional Block
x = layers.Conv2D(filters = 160, kernel_size= (3, 3), strides=(1,1),
                  padding='same', activation='relu', name ='Block7_CNN')(x)
x = layers.MaxPooling2D(pool_size=(2, 2), strides=None, padding="valid" )(x)


# Classification Conv Layer (1x1 Conv instead of Dense)
x = layers.Conv2D(filters=2, kernel_size=(1,1),padding='same',
                   activation='softmax',
                   name='Block8_Conv')(x)

# Global Average Pooling to produce final prediction
outputs =layers.Flatten()(x)
# outputs = layers.GlobalAveragePooling2D()(x)

# Output layer (binary classification)
# outputs = layers.Dense(1, activation='sigmoid', name ='Block4_Dense')(x)

# Create the model
model = models.Model(inputs=inputs, outputs=outputs, name='CatDogModel')

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Model summary
model.summary()



Model: "CatDogModel"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 Input_layer (InputLayer)    [(None, 128, 128, 1)]     0         
                                                                 
 Block1_CNN (Conv2D)         (None, 128, 128, 10)      100       
                                                                 
 Block1_MaxPool (MaxPooling2  (None, 64, 64, 10)       0         
 D)                                                              
                                                                 
 Block2_CNN (Conv2D)         (None, 64, 64, 20)        1820      
                                                                 
 max_pooling2d (MaxPooling2D  (None, 32, 32, 20)       0         
 )                                                               
                                                                 
 Block3_CNN (Conv2D)         (None, 32, 32, 40)        

In [13]:
plot_model(model, to_file='model_diagram.png', show_shapes=True, show_layer_names=True)

AttributeError: module 'pydot' has no attribute 'InvocationException'